# Translation Task

[video](https://www.youtube.com/watch?v=ISNdQcPhsts)

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from torch.utils.tensorboard import SummaryWriter

import models.deep_learning.architectures.transformer.tasks.translation as trn

load_dotenv()
device = trn.get_device()

## Config

In [ ]:
CONFIG = trn.Config(
    batch_size=8,
    num_epochs=50,
    lr=1e-4,
    src_seq_len=350,
    tgt_seq_len=350,
    d_model=512,
    dropout=0.1,
    datasource="Helsinki-NLP/opus_books",
    src_lang="en",
    tgt_lang="es",
    model_basename="tmodel_",
)


In [ ]:
writer = SummaryWriter(CONFIG.experiment_name)
Path(CONFIG.weights_folder).mkdir(parents=True, exist_ok=True)

## Load Dataset (from HuggingFace)

In [ ]:
raw_ds = trn.TranslationHFDataset.load_dataset(
    path=CONFIG.datasource,
    name=f"{CONFIG.src_lang}-{CONFIG.tgt_lang}",
    split="train",
)

## Tokenization

In [ ]:
import shutil

src_file = Path(CONFIG.tokenizer_src_file)
tgt_file = Path(CONFIG.tokenizer_tgt_file)

for p in (src_file, tgt_file):
    # Clean up accidental directories created in earlier runs.
    if p.exists() and p.is_dir():
        shutil.rmtree(p)
    p.parent.mkdir(parents=True, exist_ok=True)

tokenizer_src = trn.get_or_build_tokenizer(src_file, raw_ds, CONFIG.src_lang)
tokenizer_tgt = trn.get_or_build_tokenizer(tgt_file, raw_ds, CONFIG.tgt_lang)

In [ ]:
src_file

In [ ]:
src_file = Path(CONFIG.tokenizer_src_file)
tgt_file = Path(CONFIG.tokenizer_tgt_file)
src_file.parent.mkdir(parents=True, exist_ok=True)
tokenizer_src = trn.get_or_build_tokenizer(src_file, raw_ds, CONFIG.src_lang)
tokenizer_tgt = trn.get_or_build_tokenizer(tgt_file, raw_ds, CONFIG.tgt_lang)

## Create dataloaders

In [ ]:
train_dataloader, val_dataloader = trn.create_dataloaders(
    raw_ds, tokenizer_src, tokenizer_tgt, CONFIG
)

## Create model

In [ ]:
model = trn.Translator(
    src_vocab_size=tokenizer_src.get_vocab_size(),
    tgt_vocab_size=tokenizer_tgt.get_vocab_size(),
    dropout=CONFIG.dropout,
    src_max_length=CONFIG.src_seq_len,
    tgt_max_length=CONFIG.tgt_seq_len,
    embed_size=CONFIG.d_model,
).to(device)

## Train the model

In [ ]:
trn.train(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    tokenizer_src=tokenizer_src,
    tokenizer_tgt=tokenizer_tgt,
    device=device,
    config=CONFIG,
    writer=writer,
)